# Weight-enumerator examples

Apply Theorem 4.1 at $q=5,7$ using the exports from the reducible and genus-zero through genus-three notebooks. Run those notebooks first.

The coefficient indexed by $z$ counts codewords with $z$ rational zeros and contributes $a_zX^zY^{q^2+q+1-z}$.

## Inputs

Load the numerical exports from `../results/`.

In [1]:
from pathlib import Path
from collections import defaultdict
import json

result_dir = Path("../results")
def read_result(name, required=True):
    path = result_dir / name
    if not path.exists():
        if required:
            raise FileNotFoundError("Run the producer notebook first: " + str(path))
        return {}
    return json.loads(path.read_text())["fields"]

incidence_data = read_result("reducible_incidence.json")
genus_zero_data = read_result("genus_zero.json")
genus_one_data = read_result("genus_one.json")
genus_two_data = read_result("genus_two.json", required=False)
genus_three_data = read_result("genus_three.json", required=False)

## Line and cubic

Use [Kaplan, *Weight Enumerators of Reed–Muller Codes from Cubic Curves and Their Duals*, §5.1, Propositions 3–5 and equations (4)–(7)](https://doi.org/10.1090/conm/722/14535). The affine-cubic coefficients include the scalar factor. Multiplication by $(q^2+q+1)X^{q+1}$ adds the line. The smooth cubic term uses the elliptic distribution from `genus_1.ipynb`.

In [2]:
def line_cubic_counts(q0, elliptic):
    q0 = ZZ(q0)
    plane_size = q0^2+q0+1
    answer = defaultdict(QQ)
    def add(affine_zeros, coefficient):
        answer[ZZ(q0+1+affine_zeros)] += plane_size*coefficient
    prefactor = (q0-1)*(q0^3-q0)*q0^2
    for zeros,value in [(q0+1,(q0+1)*(q0-1)/3),
                        (q0,(q0^2-q0+4)/2),(q0-1,2*q0-1),
                        (q0-2,(q0-1)*(q0-2)/6)]:
        add(zeros,prefactor*value)
    prefactor = (q0-1)*(q0^3-q0)*(q0^3-q0^2)/2
    for zeros,value in [(q0,q0*(q0+1)/3),(q0-1,(q0^2-q0+6)/2),
                        (q0-2,2*q0-3),(q0-3,(q0-2)*(q0-3)/6),
                        (q0+2,q0*(q0-1)/3),(q0+1,q0*(q0-1)/2),
                        (q0,2*q0+1),(q0-1,q0*(q0-1)/6)]:
        add(zeros,prefactor*value)
    prefactor = q0^3*(q0-1)^2*(q0^2-1)
    for trace,weight in elliptic.items():
        t,E = ZZ(trace),QQ(weight)
        alpha = [(q0+1-t,(q0^2+q0*t+t^2-q0+t)/3),
                 (q0-t,(q0^2-t^2+q0+t+2)/2),
                 (q0-1-t,q0-t),(q0-2-t,(q0-t)*(q0-t-1)/6)]
        for zeros,value in alpha:
            add(zeros,prefactor*E*value)
    return dict(answer)

## Assembly

The multipliers are $q-1$ for projective incidence counts, $(q-1)\#\mathrm{PGL}_3/\#\mathrm{PGL}_2$ for fixed-$\mathbf P^1$ series, and $(q-1)\#\mathrm{PGL}_3$ for weighted positive-genus counts.

In [3]:
def assemble_example(q0, smooth_distribution=None):
    q0 = ZZ(q0)
    key = str(q0)
    length = q0^2+q0+1
    pgl2 = q0*(q0^2-1)
    scale = (q0-1)*q0^3*(q0^3-1)*(q0^2-1)
    families = {}
    reducible = defaultdict(QQ)
    for z,value in incidence_data[key].items():
        reducible[ZZ(z)] += (q0-1)*QQ(value)
    for z,value in line_cubic_counts(q0,genus_one_data[key]["elliptic"]).items():
        reducible[z] += value
    families["reducible"] = dict(reducible)
    families["genus 0"] = {q0+1-ZZ(d):scale*QQ(value)/pgl2
                           for d,value in genus_zero_data[key].items()}
    g1 = defaultdict(QQ)
    for t,d,value in genus_one_data[key]["coefficients"]:
        g1[q0+1-ZZ(t)-ZZ(d)] += scale*QQ(value)
    families["genus 1"] = dict(g1)
    missing = []
    if key in genus_two_data:
        families["genus 2"] = {ZZ(z):scale*QQ(value)
                               for z,value in genus_two_data[key]["coefficients"].items()}
    else:
        missing.append("genus-two distribution, run genus_2.ipynb at this q")
    if smooth_distribution is None and key in genus_three_data:
        smooth_distribution = genus_three_data[key]["S"]
    if smooth_distribution is None:
        missing.append("smooth quartics S_q(n)")
    else:
        if key not in genus_two_data:
            raise ValueError("A complete example also needs the genus-two distribution")
        families["smooth"] = {ZZ(z):scale*QQ(value) for z,value in smooth_distribution.items()}
    known = defaultdict(QQ)
    known[length] = 1
    for family,hist in families.items():
        for z,value in hist.items():
            if value == 0:
                continue
            assert 0 <= z <= length and value >= 0 and value.denominator() == 1, (family,z,value)
            known[z] += value
    assert sum(known.values()) <= q0^15
    if not missing:
        assert sum(known.values()) == q0^15
    return families, dict(known), missing

## Examples at $q=5,7$

Both saved examples include every contribution and the zero word. The coefficients sum to $q^{15}$.

In [4]:
P.<X,Y> = PolynomialRing(ZZ)
examples = {}
for q0 in [ZZ(5),ZZ(7)]:
    families,known,missing = assemble_example(q0)
    length = q0^2+q0+1
    columns = ["reducible","genus 0","genus 1","genus 2","smooth"]
    zeros = sorted(z for z,value in known.items() if value)
    rows = [(z,)+tuple(families[name].get(z,0) if name in families else "uncomputed" for name in columns)
            +(known[z],) for z in zeros]
    print("q =",q0,"and code length =",length)
    print(table(rows,header_row=["zeros z"]+columns+["total, including zero word"]))
    polynomial = sum(ZZ(value)*X^z*Y^(length-z) for z,value in known.items() if value)
    print("W_known =" if missing else "W =")
    print(polynomial)
    print("Missing:", ", ".join(missing) if missing else "none")
    print("Codewords:",sum(known.values()))
    if missing:
        print("Codewords in the missing contribution:",q0^15-sum(known.values()))
    examples[q0] = (families,known,missing)
payload = {'fields':{str(q0):{
    'length':str(q0^2+q0+1),
    'coefficients':{str(n):str(value) for n,value in known.items() if value},
    'codewords':str(sum(known.values())),
    'missing':missing,
} for q0,(_,known,missing) in examples.items()}}
(result_dir / 'weight_enumerator.json').write_text(json.dumps(payload,indent=2)+'\n')


q = 5 and code length = 31
  zeros z   reducible   genus 0     genus 1     genus 2      smooth       total, including zero word
├─────────┼───────────┼───────────┼───────────┼────────────┼────────────┼────────────────────────────┤
  0         5890000     0           0           0            25110000     31000000
  1         6994220     0           558000      7502000      220968000    236022220
  2         4743000     0           44826000    61008000     757206000    867783000
  3         1860000     62000       97464000    216132000    1821808000   2137326000
  4         620000      8060000     148428000   440200000    3098667000   3695975000
  5         0           59520000    145266000   762600000    4073028000   5040414000
  6         1909724     107756000   209622000   935208000    4160262000   5414757724
  7         9579000     54622000    170562000   1068756000   3576408000   4879927000
  8         22878000    8556000     164796000   868992000    2507140500   3572362500
  9     

Other field sizes require all four input contributions and a complete smooth distribution. `assemble_example` checks that a complete enumerator has total $q^{15}$.